In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import requests

In [2]:
df = pd.read_pickle(r"E:\Data Science\Projects\AI Teaching Assistant\embeddings.pkl")

OLLAMA_URL = "http://localhost:11434/api/embed"
MODEL_NAME = "bge-m3"
BATCH_SIZE = 16

In [3]:
def create_embedding(text_list):
    r = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "input": text_list
        },
        timeout=300
    )

    r.raise_for_status()

    response = r.json()

    if "embeddings" not in response:
        raise Exception(
            f"Embeddings not found in response.\nResponse: {response}"
        )

    return response["embeddings"]

In [4]:
question = input("Enter your question: ")
question_embedding = create_embedding([question])[0]

In [5]:
similarities = cosine_similarity(np.vstack(df['embedding'].values), [question_embedding]).flatten()
print(f"For question '{question}', similarities with existing embeddings: {similarities}")
print(f"Indices of top 5 most similar embeddings: {similarities.argsort()[-5:][::-1]}")  # Print indices of top 5 most similar embeddings

For question 'Where is forward propagation taught in this course?', similarities with existing embeddings: [0.30709545 0.44429846 0.37912803 ... 0.42248905 0.41230635 0.33294416]
Indices of top 5 most similar embeddings: [11887 10251  7766 14015 11186]


In [6]:
df.iloc[similarities.argsort()[-5:][::-1]]  # Display the top 5 most similar embeddings

,number,title,start,end,text,chunk_id,embedding
11887,15,Backpropagation in Deep Learning ｜ Part 1 ｜ Th...,397.0,398.0,What is forward propagation?,11887,"[-0.04050671, -0.025751082, -0.020548305, 0.00..."
10251,14,Loss Functions in Deep Learning ｜ Deep Learnin...,512.0,515.0,What do you do by doing forward propagation?,10251,"[-0.049695373, -0.016012875, -0.029102268, 0.0..."
7766,10,Forward Propagation ｜ How a neural network pre...,81.0,84.0,So in this video I will teach you forward pro...,7766,"[-0.057827316, -0.0027992825, -0.030756606, 0...."
14015,16,Backpropagation Part 2 ｜ The How ｜ Complete De...,1379.0,1381.0,forward propagation,14015,"[-0.037444215, 0.0005618095, -0.027239965, -0...."
11186,14,Loss Functions in Deep Learning ｜ Deep Learnin...,2467.0,2469.0,forward propagation,11186,"[-0.037444215, 0.0005618095, -0.027239965, -0...."


In [7]:
new_df = df.iloc[similarities.argsort()[-5:][::-1]]
dff = new_df[['number', 'title', 'start', 'end', 'text']]

In [8]:
prompt = f"""
You are an AI assistant helping students navigate a Deep Learning course.

Below are video transcript chunks. Each chunk contains:
- video_number
- video_title
- start (seconds)
- end (seconds)
- transcript text

Video Chunks:
{dff.to_json(orient="records")}

--------------------------------------------------

Student Question:
"{question}"

Your task:

1. Find all transcript chunks that are relevant to the student's question.
2. Match using semantic meaning, not just exact keyword matching.
   - Include synonyms and related concepts whenever appropriate.
3. Group results by video.
4. For each relevant video, provide:
   - Video Number
   - Video Title
   - Timestamp in MM:SS format
   - A very short description (5–15 words) of what is explained there.
5. Rank results from most relevant to least relevant.
6. If multiple nearby chunks belong to the same explanation, merge them into one timestamp range.
7. If only a brief mention exists, explicitly state "Brief mention".
8. If a concept is explained in depth across multiple timestamps in the same video, mention each important section.

Output format:

Video <number> – <title>
• MM:SS–MM:SS — <short description>

Video <number> – <title>
• MM:SS — Brief mention

Rules:
- Keep the response under 120 words.
- Do not use end time in the output.
- Do not invent timestamps or explanations.
- Only use the provided transcript chunks.
- Convert starting seconds to MM:SS format.
- If no relevant content exists, respond exactly:
No content found
- Do not ask follow-up questions.
- Do not add introductory or concluding text.
"""

promptt = f"""
Video transcript chunks:
{dff.to_json(orient="records")}

Question: "{question}"

Find all chunks relevant to the question (semantic match, not just keywords).

Return only:
The Relevant Videos for your question, "{question}":
Video <number> - <title>
• MM:SS : <A short enhanced description>

Group nearby chunks from the same video. Sort by relevance. Convert starting seconds to MM:SS. Use only the given data. If no relevant chunk exists, reply exactly:
No content found, do not use words like chunk, section, or part instead use word "video". Do not invent timestamps or explanations.
"""

In [9]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

def get_api_key():
    """Get API key from environment variables"""
    # Try GEMINI_API_KEY first
    api_key = os.getenv("GEMINI_API_KEY")
    if api_key:
        return api_key
    return None

api_key = get_api_key()
if not api_key:
    raise ValueError("API key not found. Please check your .env file.")

C:\Users\shree\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash') 
model.start_chat(history=[])
# Send message to the model
response = model.generate_content(prompt)
print(response.text)

Video 10 – Forward Propagation \uff5c How a neural network predicts output\uff1f
• 01:21 — This video explicitly states it will teach forward propagation.

Video 15 – Backpropagation in Deep Learning \uff5c Part 1 \uff5c The What\uff1f
• 06:37 — The video asks "What is forward propagation?" implying an explanation.

Video 14 – Loss Functions in Deep Learning \uff5c Deep Learning \uff5c CampusX
• 08:32 — The video asks about the actions involved in forward propagation.
• 41:07 — Brief mention.

Video 16 – Backpropagation Part 2 \uff5c The How \uff5c Complete Deep Learning Playlist
• 22:59 — Brief mention.
